<div style="background: linear-gradient(135deg, #0d0d0d 0%, #1a0025 50%, #0d0011 100%); padding: 30px; border-radius: 15px; text-align: center; border: 2px solid #8b0000; box-shadow: 0 0 30px rgba(139,0,0,0.5);">
  <h1 style="color: #ff4444; font-size: 2.5em; margin: 0; text-shadow: 0 0 20px #ff0000;">&#127908; Elbek Studiya RVC</h1>
  <h2 style="color: #cc44ff; font-size: 1.5em; margin: 5px 0;">Voice Model Trainer</h2>
  <p style="color: #888; font-size: 0.9em;">Gradio UI + Ngrok Tunnel | Powered by Bebra777228/TrainVocModel-EN</p>
</div>

In [ ]:
#@title **1-BOLIM: O'rnatish (1 marta ishga tushiring)**#@markdown GPU yoqilgan bo'lishi kerak! Runtime > Change runtime type > T4 GPUimport os, sys, torchfrom google.colab import drivefrom IPython.display import clear_output# --- GPU ---print("GPU tekshirilmoqda...")if not torch.cuda.is_available():    raise Exception("\033[91mGPU topilmadi! Runtime > Change runtime type > T4 GPU\033[0m")print(f"\033[92mGPU: {torch.cuda.get_device_name(0)}\033[0m")# --- GOOGLE DRIVE ---if not os.path.isdir("/content/drive"):    drive.mount("/content/drive")ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"REPO_BRANCH = "main"GITHUB_RAW = f"https://raw.githubusercontent.com/Bebra777228/TrainVocModel-EN/refs/heads/{REPO_BRANCH}"if not os.path.exists("/content/dataset"):    os.makedirs("/content/dataset")# --- CLONE ---if not os.path.isdir(ROOT_DIR):    print("Repo yuklanmoqda...")    os.system(f"git clone https://github.com/Bebra777228/TrainVocModel-EN -b {REPO_BRANCH} {ROOT_DIR} &> /dev/null")os.chdir(ROOT_DIR)clear_output()print("O'rnatish boshlandi (~6 daqiqa)...\n")# --- Python 3.10 ---print("[1/6] Python 3.10...")os.system("sudo apt update &> /dev/null && sudo apt install python3.10 -y &> /dev/null")os.system("sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 &> /dev/null")os.system("sudo update-alternatives --set python3 /usr/bin/python3.10 &> /dev/null")os.system("curl -sS https://bootstrap.pypa.io/get-pip.py | python3 &> /dev/null")sys.path.append("/usr/local/lib/python3.10/dist-packages")# --- Skriptlar ---print("[2/6] Fayllar yuklanmoqda...")scripts = [    "requirements.txt", "download_files.py", "configs/config.py",    "infer/lib/rmvpe.py", "infer/lib/slicer.py", "infer/lib/train/utils.py",    "infer/modules/train/train.py", "infer/modules/vc/infer_cli.py",    "infer/lib/train/process_ckpt.py", "infer/modules/train/extract_f0.py",    "infer/modules/train/preprocess.py", "infer/modules/train/extract_index.py",    "infer/modules/train/extract_feature.py",]for s in scripts:    dest = f"{ROOT_DIR}/{s}"    os.makedirs(os.path.dirname(dest), exist_ok=True)    os.system(f"wget {GITHUB_RAW}/{s} -O {dest} &> /dev/null")# --- Pip ---print("[3/6] Kutubxonalar...")os.system("pip install --upgrade pip==24.0 &> /dev/null")os.system("pip install --no-cache-dir -qq -r requirements.txt &> /dev/null")# --- aria2, ffmpeg ---print("[4/6] aria2, ffmpeg...")os.system("sudo apt install -y aria2 ffmpeg &> /dev/null")# --- HuBERT ---print("[5/6] HuBERT modeli...")os.system('python download_files.py "./assets/" "contentvec_base" &> /dev/null')# --- Gradio + pyngrok ---print("[6/6] Gradio + pyngrok...")os.system("pip install -qq gradio pyngrok &> /dev/null")# --- Tozalash ---os.system("rm -rf /content/sample_data")os.system(f"rm -rf {ROOT_DIR}/.git {ROOT_DIR}/notebooks")clear_output()print("\033[92m" + "=" * 50)print("  TAYYOR! Endi 2-BOLIMNI ishga tushiring!")print("=" * 50 + "\033[0m")

In [ ]:
#@title **2-BOLIM: Gradio UI ishga tushirish (ngrok link)**#@markdown ngrok tokeningizni kiriting (https://dashboard.ngrok.com/get-started/your-authtoken)ngrok_token = "" #@param {type:"string"}import osimport sysimport globimport jsonimport shutilimport zipfileimport pathlibimport tracebackimport threadingfrom random import shufflefrom subprocess import PIPE, STDOUT, Popenfrom urllib.parse import urlparseROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"os.chdir(ROOT_DIR)# ===== GRADIO IMPORT =====import gradio as gr# ===== NGROK SETUP =====use_ngrok = Falseif ngrok_token and ngrok_token.strip():    try:        from pyngrok import ngrok as pyngrok_mod        pyngrok_mod.set_auth_token(ngrok_token.strip())        use_ngrok = True        print("ngrok token qabul qilindi!")    except Exception as e:        print(f"ngrok xatosi: {e}")        use_ngrok = False# ===== TRENING FUNKSIYALARI =====training_log_text = ""is_training = Falsedef full_train(model_name, dataset_audio_files, sample_rate, f0_method, index_algorithm,               pretrain_type, epochs, save_epoch, batch_size, progress=gr.Progress()):    global training_log_text, is_training    if not model_name or not model_name.strip():        return "XATO: Model nomini kiriting!", "", None    if not dataset_audio_files or len(dataset_audio_files) == 0:        return "XATO: Ovoz fayllarini yuklang!", "", None    model_name = model_name.strip().replace(" ", "_")    is_training = True    training_log_text = ""    log_lines = []    def add_log(msg):        nonlocal log_lines        log_lines.append(msg)        if len(log_lines) > 200:            log_lines = log_lines[-200:]    try:        os.chdir(ROOT_DIR)        now_dir = os.getcwd()        # --- 1. Dataset papka tayyorlash ---        add_log("=" * 40)        add_log("1/6: Dataset tayyorlanmoqda...")        progress(0.05, desc="Dataset tayyorlanmoqda...")        dataset_folder = f"/content/dataset_{model_name}"        if os.path.exists(dataset_folder):            shutil.rmtree(dataset_folder)        os.makedirs(dataset_folder, exist_ok=True)        for audio_file in dataset_audio_files:            src = audio_file  # gradio fayl yo'li            fname = os.path.basename(src)            dst = os.path.join(dataset_folder, fname)            shutil.copy2(src, dst)            add_log(f"  Fayl nusxalandi: {fname}")        add_log(f"  Jami: {len(dataset_audio_files)} ta fayl")        # --- 2. Preprocess ---        add_log("=" * 40)        add_log("2/6: Audio qayta ishlanmoqda (preprocess)...")        progress(0.15, desc="Preprocess...")        sr_int = int(sample_rate.rstrip("k")) * 1000        exp_dir = f"./logs/{model_name}"        os.makedirs(exp_dir, exist_ok=True)        preprocess_cmd = f"python {ROOT_DIR}/infer/modules/train/preprocess.py {dataset_folder} {sr_int} 2 ./logs/{model_name} False 3.0"        ret = os.system(preprocess_cmd)        add_log(f"  Preprocess tugadi (code: {ret})")        # --- 3. F0 Extraction ---        add_log("=" * 40)        add_log(f"3/6: F0 chiqarilmoqda ({f0_method})...")        progress(0.30, desc="F0 extraction...")        f0_cmd = f"python {ROOT_DIR}/infer/modules/train/extract_f0.py 1 0 0 ./logs/{model_name} False {f0_method}"        ret = os.system(f0_cmd)        add_log(f"  F0 tugadi (code: {ret})")        # --- 4. Feature Extraction ---        add_log("=" * 40)        add_log("4/6: Xususiyatlar chiqarilmoqda (feature)...")        progress(0.45, desc="Feature extraction...")        feat_cmd = f"python {ROOT_DIR}/infer/modules/train/extract_feature.py 1 0 ./logs/{model_name} v2 False"        ret = os.system(feat_cmd)        add_log(f"  Feature tugadi (code: {ret})")        # --- 5. Index ---        add_log("=" * 40)        add_log("5/6: Index yaratilmoqda...")        progress(0.55, desc="Index extraction...")        idx_cmd = f"python {ROOT_DIR}/infer/modules/train/extract_index.py ./logs/{model_name} v2 {index_algorithm}"        ret = os.system(idx_cmd)        add_log(f"  Index tugadi (code: {ret})")        # --- 6. Pretrain yuklash + Training ---        add_log("=" * 40)        add_log(f"6/6: Trening ({pretrain_type} pretrain, {epochs} epoch)...")        progress(0.60, desc="Pretrain yuklanmoqda...")        param_aria = "--console-log-level=error -c -x 16 -s 16 -k 1M"        hugg_pret = ("ht" + "tps:/" + "/hug" + "gin" + "gfa" + "ce.co" + "/Poli" + "tree"                     + "s/RV" + "C_res" + "ourc" + "es/re" + "solv" + "e/ma" + "in/pret"                     + "rain" + "ed/v2")        pretrain_outpath = "/content/pretrained_models"        os.system(f"rm -rf {pretrain_outpath}")        os.makedirs(pretrain_outpath, exist_ok=True)        models_map = {            "Default": [                (f"{sample_rate}/Default/f0D{sample_rate}.pth", "default_D.pth"),                (f"{sample_rate}/Default/f0G{sample_rate}.pth", "default_G.pth"),            ],            "Snowie v3.1": [                (f"{sample_rate}/Snowie/D_SnowieV3.1_{sample_rate}.pth", "Snowie_D.pth"),                (f"{sample_rate}/Snowie/G_SnowieV3.1_{sample_rate}.pth", "Snowie_G.pth"),            ],            "TITAN-Medium": [                (f"{sample_rate}/TITAN/D-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_D.pth"),                (f"{sample_rate}/TITAN/G-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_G.pth"),            ],            "KLM v4.3 x3": [                (f"{sample_rate}/KLM/D_KLM43_X3_{sample_rate}.pth", "KLM_D.pth"),                (f"{sample_rate}/KLM/G_KLM43_X3_{sample_rate}.pth", "KLM_G.pth"),            ],        }        pret = models_map[pretrain_type]        for f in pret:            os.system(f"aria2c {param_aria} {hugg_pret}/{f[0]} -d {pretrain_outpath} -o {f[1]} &> /dev/null")        G_file = f"{pretrain_outpath}/{pret[1][1]}"        D_file = f"{pretrain_outpath}/{pret[0][1]}"        add_log(f"  Pretrain tayyor: G={pret[1][1]}, D={pret[0][1]}")        # --- filelist.txt ---        f0_dir      = f"{now_dir}/logs/{model_name}/2a_f0"        f0nsf_dir   = f"{now_dir}/logs/{model_name}/2b-f0nsf"        gt_wavs_dir = f"{now_dir}/logs/{model_name}/0_gt_wavs"        feature_dir = f"{now_dir}/logs/{model_name}/3_feature768"        names = (            set([n.split(".")[0] for n in os.listdir(gt_wavs_dir)])            & set([n.split(".")[0] for n in os.listdir(feature_dir)])            & set([n.split(".")[0] for n in os.listdir(f0_dir)])            & set([n.split(".")[0] for n in os.listdir(f0nsf_dir)])        )        opt = []        for name in names:            opt.append(                f"{gt_wavs_dir}/{name}.wav|"                f"{feature_dir}/{name}.npy|"                f"{f0_dir}/{name}.wav.npy|"                f"{f0nsf_dir}/{name}.wav.npy|0"            )        for _ in range(2):            opt.append(                f"{now_dir}/logs/mute/0_gt_wavs/mute{sample_rate}.wav|"                f"{now_dir}/logs/mute/3_feature768/mute.npy|"                f"{now_dir}/logs/mute/2a_f0/mute.wav.npy|"                f"{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|0"            )        shuffle(opt)        with open(f"{now_dir}/logs/{model_name}/filelist.txt", "w") as fl:            fl.write("\n".join(opt))        # --- config.json ---        if sample_rate == "40k":            config_path = f"configs/v1/{sample_rate}.json"        else:            config_path = f"configs/v2/{sample_rate}.json"        config_save = f"{now_dir}/logs/{model_name}/config.json"        if not os.path.exists(config_save):            with open(config_path, "r") as cf:                cdata = json.load(cf)                cdata["train"]["fp16_run"] = True                cdata["train"]["log_interval"] = 10            with open(config_save, "w", encoding="utf-8") as cf:                json.dump(cdata, cf, ensure_ascii=False, indent=4, sort_keys=True)                cf.write("\n")        # --- TRAIN ---        add_log("  Trening boshlanmoqda...")        progress(0.65, desc=f"Trening: 0/{epochs} epoch...")        train_cmd = (            f"python {ROOT_DIR}/infer/modules/train/train.py "            f"-e '{model_name}' "            f"-sr {sample_rate} "            f"-bs {batch_size} "            f"-te {epochs} "            f"-se {save_epoch} "            f"-pg {G_file} "            f"-pd {D_file} "            f"-o AdamW"        )        p = Popen(train_cmd, shell=True, cwd=now_dir, stdout=PIPE, stderr=STDOUT,                  bufsize=1, universal_newlines=True)        for line in p.stdout:            line = line.strip()            if line:                add_log(line)                # epoch progress                if "Epoch" in line or "epoch" in line:                    try:                        parts = line.split()                        for pt in parts:                            if pt.isdigit():                                ep = int(pt)                                if ep <= epochs:                                    pct = 0.65 + 0.30 * (ep / epochs)                                    progress(pct, desc=f"Trening: {ep}/{epochs} epoch...")                                break                    except:                        pass        p.wait()        add_log(f"  Train process tugadi (code: {p.returncode})")        # --- ZIP ---        add_log("=" * 40)        add_log("ZIP yaratilmoqda...")        progress(0.96, desc="ZIP yaratilmoqda...")        pth_files = []        weights_dir = f"{ROOT_DIR}/assets/weights"        if os.path.isdir(weights_dir):            pth_files += glob.glob(f"{weights_dir}/{model_name}.pth")            pth_files += glob.glob(f"{weights_dir}/{model_name}_e*.pth")        logs_dir = f"{ROOT_DIR}/logs/{model_name}"        if os.path.isdir(logs_dir):            pth_files += glob.glob(f"{logs_dir}/**/*.pth", recursive=True)        pth_files = list(dict.fromkeys(pth_files))        index_files = []        if os.path.isdir(logs_dir):            index_files += glob.glob(f"{logs_dir}/added_{model_name}_v2.index")            index_files += glob.glob(f"{logs_dir}/added_*.index")            index_files += glob.glob(f"{logs_dir}/*.index")        index_files = list(dict.fromkeys(index_files))        main_pth = None        if pth_files:            exact = [f for f in pth_files if os.path.basename(f) == f"{model_name}.pth"]            main_pth = exact[0] if exact else sorted(pth_files)[-1]        main_index = index_files[0] if index_files else None        zip_path = f"/content/{model_name}_model.zip"        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:            if main_pth:                zf.write(main_pth, os.path.basename(main_pth))                add_log(f"  ZIP: {os.path.basename(main_pth)}")            if main_index:                zf.write(main_index, os.path.basename(main_index))                add_log(f"  ZIP: {os.path.basename(main_index)}")        zip_size = os.path.getsize(zip_path) / (1024 * 1024)        add_log(f"  ZIP tayyor: {zip_size:.1f} MB")        add_log("=" * 40)        add_log("TAYYOR! Model muvaffaqiyatli qilindi!")        progress(1.0, desc="Tayyor!")        is_training = False        status = f"TAYYOR! .pth: {os.path.basename(main_pth) if main_pth else 'topilmadi'} | .index: {os.path.basename(main_index) if main_index else 'topilmadi'} | ZIP: {zip_size:.1f} MB"        return status, "\n".join(log_lines), zip_path    except Exception as e:        is_training = False        err = traceback.format_exc()        add_log(f"\nXATOLIK:\n{err}")        return f"XATO: {str(e)}", "\n".join(log_lines), None# ===== GRADIO UI =====custom_css = """.gradio-container {    background: linear-gradient(180deg, #0d0d0d 0%, #1a0020 50%, #0d0011 100%) !important;    font-family: 'Segoe UI', sans-serif;}.gr-button-primary {    background: linear-gradient(135deg, #8b0000 0%, #cc0000 100%) !important;    border: none !important;    font-size: 18px !important;    padding: 12px 30px !important;}.gr-button-primary:hover {    background: linear-gradient(135deg, #cc0000 0%, #ff2222 100%) !important;}footer { display: none !important; }"""with gr.Blocks(css=custom_css, title="Elbek Studiya RVC Trainer", theme=gr.themes.Base(    primary_hue="red",    secondary_hue="purple",    neutral_hue="gray",    font=["Segoe UI", "sans-serif"],)) as app:    gr.HTML(        '<div style="text-align:center; padding: 20px;">'        '<h1 style="color:#ff4444; font-size:2.5em; margin:0; text-shadow: 0 0 20px #ff0000;">'        '&#127908; Elbek Studiya RVC</h1>'        '<h3 style="color:#cc44ff;">Voice Model Trainer</h3>'        '<p style="color:#666;">Ovoz modeli yaratish uchun professional vosita</p>'        '</div>'    )    with gr.Tab("Train / Trening"):        with gr.Row():            with gr.Column(scale=1):                gr.Markdown("### Model Sozlamalari")                model_name_input = gr.Textbox(                    label="Model nomi",                    placeholder="Masalan: MyVoice",                    value="MyVoice",                    info="Model uchun nom kiriting (bo'sh joy ishlatmang)"                )                sample_rate_dd = gr.Dropdown(                    choices=["32k", "40k", "48k"],                    value="40k",                    label="Sample Rate",                    info="40k - ko'p hollarda eng yaxshi"                )                f0_method_dd = gr.Dropdown(                    choices=["rmvpe", "harvest"],                    value="rmvpe",                    label="F0 Method",                    info="rmvpe - tezroq va aniqroq"                )                index_algo_dd = gr.Dropdown(                    choices=["Auto", "Faiss", "KMeans"],                    value="Auto",                    label="Index Algorithm"                )                pretrain_dd = gr.Dropdown(                    choices=["Default", "Snowie v3.1", "TITAN-Medium", "KLM v4.3 x3"],                    value="Default",                    label="Pretrained Model",                    info="Default - ko'p hollarda yaxshi ishlaydi"                )            with gr.Column(scale=1):                gr.Markdown("### Trening Sozlamalari")                epochs_slider = gr.Slider(                    minimum=50, maximum=2000, value=500, step=50,                    label="Epoch soni",                    info="500-1000 tavsiya etiladi"                )                save_epoch_slider = gr.Slider(                    minimum=10, maximum=500, value=50, step=10,                    label="Har necha epochda saqlash"                )                batch_size_slider = gr.Slider(                    minimum=2, maximum=32, value=8, step=2,                    label="Batch Size",                    info="T4 GPU uchun 8 yaxshi"                )                gr.Markdown("### Ovoz Fayllarini Yuklash")                audio_upload = gr.File(                    label="Audio fayllar (WAV, FLAC, MP3)",                    file_count="multiple",                    file_types=["audio"],                    type="filepath",                )        with gr.Row():            train_btn = gr.Button(                "TRENING BOSHLASH",                variant="primary",                size="lg",            )        with gr.Row():            status_output = gr.Textbox(label="Holat", interactive=False, lines=2)        with gr.Row():            log_output = gr.Textbox(label="Log", interactive=False, lines=15, max_lines=25)        with gr.Row():            zip_output = gr.File(label="ZIP yuklab olish (.pth + .index)")        train_btn.click(            fn=full_train,            inputs=[                model_name_input, audio_upload, sample_rate_dd, f0_method_dd,                index_algo_dd, pretrain_dd, epochs_slider, save_epoch_slider,                batch_size_slider,            ],            outputs=[status_output, log_output, zip_output],        )    with gr.Tab("Ma'lumot"):        gr.Markdown("""### Qanday ishlatish:1. **Model nomi** kiriting2. **Ovoz fayllarini** yuklang (WAV/FLAC/MP3, 10+ daqiqa tavsiya)3. **Epoch** ni tanlang (500-1000)4. **TRENING BOSHLASH** tugmasini bosing5. Trening tugaganda **ZIP** avtomatik yuklanadi### Sozlamalar:- **Sample Rate**: 40k - universal, 48k - yuqori sifat- **F0 Method**: rmvpe - tez va aniq, harvest - klassik- **Pretrain**: Default - eng ko'p ishlatiladigan- **Batch Size**: 8 - T4 GPU uchun optimal### Chiqish fayllari:- `.pth` - ovoz modeli- `.index` - indeks fayli (sifatni oshiradi)---*Elbek Studiya RVC Trainer v5 | Bebra777228/TrainVocModel-EN*        """)# ===== LAUNCH =====print("\n" + "=" * 50)print("  Gradio UI ishga tushmoqda...")print("=" * 50 + "\n")if use_ngrok:    app.launch(        share=False,        server_port=7860,        quiet=True,    )    from pyngrok import ngrok as pyngrok_mod    public_url = pyngrok_mod.connect(7860, "http")    print(f"\n{'=' * 50}")    print(f"  NGROK LINK: {public_url}")    print(f"{'=' * 50}\n")    print("Shu linkni oching va model qiling!")else:    app.launch(        share=True,        server_port=7860,        quiet=False,    )    print("\nGradio public link yuqorida ko'rsatilgan!")    print("(ngrok token kiritmadingiz, Gradio share link ishlatilmoqda)")